# 02 — Calibração de NSGA-III e MOEA/D

A calibração é separada do experimento final, usa apenas correlação média e sementes 1–3. SMOKE valida o espaço sem escolher parâmetros científicos.

In [ ]:
from pathlib import Path
import json, os, time, math, gc
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import ConvexHull, Delaunay, cKDTree
from scipy.stats import qmc
from math import comb
from pymoo.util.ref_dirs import get_reference_directions

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
if MODE=='FULL':
    raise RuntimeError('FULL bloqueado; não existe desbloqueio por variável de ambiente.')
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}
OUT=ROOT/'results'/'tuning'; OUT.mkdir(parents=True,exist_ok=True)

def candidates(m,budget=100000):
    return [{'m':m,'n_partitions':p,'n_directions':comb(m+p-1,p),'pop_size':comb(m+p-1,p),'at_least_two_generations':2*comb(m+p-1,p)<=budget} for p in (2,3,4)]

stage_a=pd.DataFrame([r for m in CFG['scenario_objectives'] for r in candidates(m)])
assert np.all(stage_a.n_directions==stage_a.apply(lambda r:len(get_reference_directions('das-dennis',int(r.m),n_partitions=int(r.n_partitions))),axis=1))
stage_a.to_csv(OUT/'stage_a_reference_directions.csv',index=False)

## Calibração executável e experimentos evolucionários

Executa os estágios A/B/C com checkpoints por configuração e semente; depois congela os vencedores e roda as sementes finais sob o orçamento CNBI medido.

In [ ]:
# PERMANENT EXECUTABLE EA TUNING
from itertools import product
import hashlib
SCHEMA_VERSION=2
IMPLEMENTATION_FINGERPRINT='ea-pymoo-v2'
def stable_hash(value):
    if isinstance(value,np.ndarray): payload=np.ascontiguousarray(value).tobytes()
    else: payload=json.dumps(value,sort_keys=True,separators=(',',':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()
def checkpoint_identity(scenario,method,m,seed,budget,params,A,B,stage):
    body={'schema_version':SCHEMA_VERSION,'implementation_fingerprint':IMPLEMENTATION_FINGERPRINT,'mode':MODE,'scenario':scenario,'method':method,'dimension':int(m),'seed':int(seed),'budget':int(budget),'parameters':params,'stage':stage,'config_hash':stable_hash(CFG),'anchors_hash':stable_hash(A),'rsm_hash':stable_hash(B)}; body['fingerprint']=stable_hash(body); return body
from scipy.spatial import cKDTree
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize as pymoo_minimize

GEN=ROOT/'data'/'generated'; REFS=ROOT/'data'/'reference_fronts'
if MODE=='SMOKE': GEN=GEN/'smoke'; REFS=REFS/'smoke'
TABLES=ROOT/'results'/'tables'; CK=ROOT/'results'/'checkpoints'
TABLES.mkdir(parents=True,exist_ok=True); CK.mkdir(parents=True,exist_ok=True)

def design(X):
    X=np.atleast_2d(X); x1,x2,x3=X.T
    return np.column_stack([np.ones(len(X)),x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])

XDOE=np.vstack([np.array(list(product([-1.,1.],repeat=3))),np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA]),np.zeros((5,3))]); D=design(XDOE)

class SphereRepair(Repair):
    def _do(self,problem,X,**kwargs):
        X=np.asarray(X,float); norm=np.linalg.norm(X,axis=1,keepdims=True); return X*np.minimum(1,ALPHA/np.maximum(norm,1e-15))

class RSMProblem(Problem):
    def __init__(self,B):
        super().__init__(n_var=3,n_obj=B.shape[1],xl=-ALPHA,xu=ALPHA); self.B=B; self.actual_evaluations=0
    def _evaluate(self,X,out,*args,**kwargs):
        X=SphereRepair()._do(self,X); self.actual_evaluations+=len(X); out['F']=design(X)@self.B

def fitted_rsm(A,seed):
    F=np.sum((XDOE[:,None,:]-A[None,:,:])**2,axis=2); rng=np.random.default_rng(seed); sigma=np.sqrt(F.var(0,ddof=1)*(.05/.95)); Y=F+rng.normal(0,sigma,F.shape); return np.linalg.lstsq(D,Y,rcond=None)[0]

def normalized_true_reference(scenario):
    d=np.load(REFS/f'{scenario}_pareto_reference.npz'); F=d['F']; ideal=np.asarray(d['ideal_true']); nadir=np.asarray(d['nadir_true']); amp=np.maximum(nadir-ideal,1e-12); return (F-ideal)/amp,ideal,amp

def run_ea(method,B,params,budget,seed):
    m=B.shape[1]; ref_dirs=get_reference_directions('das-dennis',m,n_partitions=int(params['n_partitions'])); pop=len(ref_dirs)
    if 2*pop>budget: return None,{'status':'BUDGET_INCOMPATIBLE','pop_size':pop,'rsm_evaluations':0}
    crossover=SBX(prob=float(params['sbx_probability']),eta=float(params['sbx_eta']),repair=SphereRepair()); mutation=PM(prob=1/3,eta=float(params['pm_eta']),repair=SphereRepair())
    if method=='NSGAIII': alg=NSGA3(ref_dirs=ref_dirs,pop_size=pop,crossover=crossover,mutation=mutation,repair=SphereRepair())
    else:
        neighbors=max(2,min(pop-1,round(float(params.get('neighbor_fraction',.2))*pop)))
        alg=MOEAD(ref_dirs=ref_dirs,n_neighbors=neighbors,prob_neighbor_mating=float(params.get('prob_neighbor_mating',.9)),crossover=crossover,mutation=mutation,repair=SphereRepair())
    generations=max(2,budget//pop); problem=RSMProblem(B); t0=time.perf_counter(); c0=time.process_time()
    res=pymoo_minimize(problem,alg,('n_gen',generations),seed=int(seed),verbose=False,save_history=False)
    X=SphereRepair()._do(problem,np.atleast_2d(res.X)); F=design(X)@B
    return (X,F),{'status':'COMPLETED','pop_size':pop,'generations':generations,'rsm_evaluations':problem.actual_evaluations,'wall_seconds':time.perf_counter()-t0,'cpu_seconds':time.process_time()-c0}

def tuning_metric(F_rsm,Rn,ideal,amp): return float(cKDTree((F_rsm-ideal)/amp).query(Rn,k=1)[0].mean())

def candidate_id(p): return '_'.join(f'{k}-{str(v).replace(".","p")}' for k,v in sorted(p.items()))

def evaluate_candidates(method,m,scenario,candidates,budget,seeds,stage):
    A=np.load(GEN/f'{scenario}_scenario.npz')['anchors']; Rn,ideal,amp=normalized_true_reference(scenario); rows=[]
    for params in candidates:
      cid=candidate_id(params)
      for seed in seeds:
        B=fitted_rsm(A,seed); identity=checkpoint_identity(scenario,method,m,seed,budget,params,A,B,stage); ck=CK/f'tuning_{MODE.lower()}_{scenario}_{method}_{stage}_{cid}_seed{seed}_{identity["fingerprint"][:12]}.json'
        cached=json.loads(ck.read_text(encoding='utf-8')) if ck.exists() else None
        if cached is not None and cached.get('identity')==identity: row={**cached,'checkpoint_reused':True}
        else:
            result,meta=run_ea(method,B,params,budget,seed); row={'method':method,'m':m,'stage':stage,'config_id':cid,'seed':seed,'identity':identity,'checkpoint_reused':False,**params,**meta}
            row['IGD']=np.nan if result is None else tuning_metric(result[1],Rn,ideal,amp); ck.write_text(json.dumps(row,indent=2),encoding='utf-8')
        rows.append(row)
    return rows

def rank_configs(rows):
    d=pd.DataFrame(rows); ok=d[d.status.eq('COMPLETED')]; return ok.groupby('config_id',as_index=False).agg(IGD_median=('IGD','median'),IGD_iqr=('IGD',lambda x:x.quantile(.75)-x.quantile(.25)),wall_median=('wall_seconds','median')).sort_values(['IGD_median','IGD_iqr','wall_median'])

def execute_tuning_and_eas():
    if not CFG['run_tuning']: return pd.DataFrame(),{}
    det=pd.read_csv(TABLES/f'{MODE.lower()}_deterministic_runs.csv'); allrows=[]; selected={}
    for m in CFG['scenario_objectives']:
      scenario=f'm{m}_medium'; measured=int(det[(det.scenario==scenario)&(det.method=='CNBI')].rsm_evaluations.median()); budget=min(measured,int(CFG.get('tuning_budget_cap',measured)))
      for method in ('NSGAIII','MOEAD'):
        base={'sbx_probability':1.,'sbx_eta':20,'pm_eta':20,'neighbor_fraction':.2,'prob_neighbor_mating':.9}
        candA=[{'n_partitions':p,**base} for p in (2,3,4)]; rowsA=evaluate_candidates(method,m,scenario,candA,budget,CFG['calibration_seeds'],'A'); allrows+=rowsA
        topA=rank_configs(rowsA).head(2).config_id.tolist(); pbest=[next(int(r['n_partitions']) for r in rowsA if r['config_id']==cid) for cid in topA]
        frac=[(pbest[i%2],(.9,1.)[(i//2)%2],(10,20,30)[i%3],(15,20,30)[(i//3+i)%3]) for i in range(12)]
        coverage=pd.DataFrame(frac,columns=['n_partitions','sbx_probability','sbx_eta','pm_eta']); assert coverage.n_partitions.value_counts().eq(6).all() and coverage.sbx_probability.value_counts().eq(6).all() and coverage.sbx_eta.value_counts().eq(4).all() and coverage.pm_eta.value_counts().eq(4).all() and len(coverage.drop_duplicates())==12
        coverage.assign(method=method,m=m).to_csv(OUT/f'stage_b_coverage_{method}_m{m}.csv',index=False)
        candB=[{'n_partitions':p,'sbx_probability':sp,'sbx_eta':se,'pm_eta':pe,'neighbor_fraction':.2,'prob_neighbor_mating':.9} for p,sp,se,pe in frac]; rowsB=evaluate_candidates(method,m,scenario,candB,budget,CFG['calibration_seeds'],'B'); allrows+=rowsB
        bestB=rank_configs(rowsB).iloc[0]; proto=next(r for r in rowsB if r['config_id']==bestB.config_id); best={k:proto[k] for k in ('n_partitions','sbx_probability','sbx_eta','pm_eta','neighbor_fraction','prob_neighbor_mating')}
        if method=='MOEAD':
            candC=[{**best,'neighbor_fraction':nf,'prob_neighbor_mating':pm} for nf in (.1,.2,.3) for pm in (.7,.9,1.)]; rowsC=evaluate_candidates(method,m,scenario,candC,budget,CFG['calibration_seeds'],'C'); allrows+=rowsC; win=rank_configs(rowsC).iloc[0]; proto=next(r for r in rowsC if r['config_id']==win.config_id); best={k:proto[k] for k in best}
        selected[f'{method}_m{m}']={'status':'COMPLETED','scenario':scenario,'budget_used_for_tuning':budget,**best}
    pd.DataFrame(allrows).to_csv(OUT/'tuning_all_results.csv',index=False); (OUT/'chosen_parameters.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    manifests=[]
    for m in CFG['scenario_objectives']:
      for level in CFG['correlation_targets']:
        scenario=f'm{m}_{level}'; A=np.load(GEN/f'{scenario}_scenario.npz')['anchors']; detm=det[(det.scenario==scenario)&(det.method=='CNBI')]
        for method,label in (('NSGAIII','NSGA-III'),('MOEAD','MOEA/D')):
          params=selected[f'{method}_m{m}']
          for seed in CFG['final_seeds']:
            fullbudget=int(detm[detm.seed.astype(int)==int(seed)].rsm_evaluations.iloc[0])
            Bfinal=fitted_rsm(A,seed); final_params={k:params[k] for k in ('n_partitions','sbx_probability','sbx_eta','pm_eta','neighbor_fraction','prob_neighbor_mating')}; identity=checkpoint_identity(scenario,method,m,seed,fullbudget,final_params,A,Bfinal,'FINAL'); ck=CK/f'{MODE.lower()}_{scenario}_seed{seed}_{method}_budget{fullbudget}_{identity["fingerprint"][:12]}.npz'; t0=time.perf_counter(); reused=False
            if ck.exists():
                dat=np.load(ck,allow_pickle=False); meta=json.loads(str(dat['metadata'])); reused=meta.get('identity')==identity; n=len(dat['X']) if reused else 0
            if not reused:
                result,meta=run_ea(method,Bfinal,params,fullbudget,seed); Xv,Fv=result; meta={**meta,'identity':identity,'budget_reference':fullbudget,'parameters':final_params}; assert meta['rsm_evaluations']<=fullbudget; np.savez_compressed(ck,X=Xv,F_rsm=Fv,success=np.ones(len(Xv),bool),metadata=json.dumps(meta)); n=len(Xv)
            manifests.append({'scenario':scenario,'seed':seed,'method':label,'status':meta['status'],'n_solutions':n,'rsm_evaluations':meta['rsm_evaluations'],'wall_seconds':meta.get('wall_seconds',time.perf_counter()-t0),'cpu_seconds':meta.get('cpu_seconds',0),'checkpoint_reused':reused,'checkpoint':ck.relative_to(ROOT).as_posix()})
    ea=pd.DataFrame(manifests); ea.to_csv(TABLES/f'{MODE.lower()}_ea_runs.csv',index=False); return ea,selected

ea_manifest,chosen_real=execute_tuning_and_eas()
if CFG['run_tuning']: print(ea_manifest.to_string(index=False))


## Validação dos finalistas

Valida os três melhores finalistas por método e dimensão com IGD, hipervolume verdadeiro, inviabilidade, variabilidade e custo, na ordem normativa.

In [ ]:
# FINALIST VALIDATION WITH ORDERED CRITERIA
from scipy.stats import qmc

def tuning_hv_qmc(Fn,m,seed,n=4096,reference=1.1):
    engine=qmc.Sobol(d=m,scramble=True,seed=seed); U=engine.random_base2(int(np.log2(n)))*reference
    dominated=np.zeros(len(U),bool)
    for start in range(0,len(U),256):
        block=U[start:start+256]; dominated[start:start+256]=np.any(np.all(Fn[:,None,:]<=block[None,:,:],axis=2),axis=0)
    p=dominated.mean(); volume=reference**m
    return float(volume*p),float(volume*np.sqrt(p*(1-p)/len(U)))

def execute_finalist_validation():
    if not CFG['run_tuning']: return pd.DataFrame()
    raw=pd.read_csv(OUT/'tuning_all_results.csv'); finalists=[]; rankings=[]; selected={}
    for m in CFG['scenario_objectives']:
      scenario=f'm{m}_medium'; A=np.load(GEN/f'{scenario}_scenario.npz')['anchors']; Rn,ideal,amp=normalized_true_reference(scenario)
      measured=pd.read_csv(TABLES/f'{MODE.lower()}_deterministic_runs.csv'); measured=int(measured[(measured.scenario==scenario)&(measured.method=='CNBI')].rsm_evaluations.median()); budget=min(measured,int(CFG.get('tuning_budget_cap',measured)))
      for method in ('NSGAIII','MOEAD'):
        stage='C' if method=='MOEAD' else 'B'; pool=raw[(raw.method==method)&(raw.m==m)&(raw.stage==stage)&(raw.status=='COMPLETED')]
        top=(pool.groupby('config_id',as_index=False).agg(IGD_pre=('IGD','median')).sort_values('IGD_pre').head(3).config_id.tolist())
        for cid in top:
          proto=pool[pool.config_id==cid].iloc[0]; params={k:proto[k] for k in ('n_partitions','sbx_probability','sbx_eta','pm_eta','neighbor_fraction','prob_neighbor_mating')}; params['n_partitions']=int(params['n_partitions'])
          for seed in CFG['calibration_seeds']:
            B=fitted_rsm(A,seed); result,meta=run_ea(method,B,params,budget,seed); Xv,Fv=result; Fn=(Fv-ideal)/amp; igd=tuning_metric(Fv,Rn,ideal,amp); hv,hv_se=tuning_hv_qmc(Fn,m,9000+m*100+int(seed)); violation=np.maximum(np.sum(Xv*Xv,axis=1)-ALPHA**2,0)
            finalists.append({'method':method,'m':m,'scenario':scenario,'config_id':cid,'seed':seed,**params,**meta,'IGD':igd,'HV':hv,'HV_se':hv_se,'infeasibility':float(np.mean(violation>1e-9))})
        fd=pd.DataFrame([r for r in finalists if r['method']==method and r['m']==m]); rank=(fd.groupby('config_id',as_index=False).agg(IGD_median=('IGD','median'),HV_median=('HV','median'),infeasibility_median=('infeasibility','median'),IGD_iqr=('IGD',lambda x:x.quantile(.75)-x.quantile(.25)),wall_median=('wall_seconds','median')).sort_values(['IGD_median','HV_median','infeasibility_median','IGD_iqr','wall_median'],ascending=[True,False,True,True,True]).reset_index(drop=True)); rank['rank']=np.arange(1,len(rank)+1); rank.insert(0,'m',m); rank.insert(0,'method',method); rankings.append(rank)
        winner=rank.iloc[0].config_id; proto=fd[fd.config_id==winner].iloc[0]; selected[f'{method}_m{m}']={'status':'COMPLETED','scenario':scenario,'budget_used_for_tuning':budget,'selection_order':['IGD_median','HV_median_desc','infeasibility_median','IGD_iqr','wall_median'],'justification':f'Primeiro colocado entre três finalistas pelo critério lexicográfico normativo: {winner}',**{k:(int(proto[k]) if k=='n_partitions' else float(proto[k])) for k in ('n_partitions','sbx_probability','sbx_eta','pm_eta','neighbor_fraction','prob_neighbor_mating')}}
    final=pd.DataFrame(finalists); ranking=pd.concat(rankings,ignore_index=True); final.to_csv(OUT/'tuning_finalists.csv',index=False); ranking.to_csv(OUT/'tuning_rankings.csv',index=False); (OUT/'chosen_parameters.json').write_text(json.dumps(selected,indent=2),encoding='utf-8'); return final

finalist_results=execute_finalist_validation()
if CFG['run_tuning']: print(f'Finalistas validados: {len(finalist_results)}')